In [1]:
import numpy as np
from scipy.stats import weibull_min
import matplotlib.pyplot as plt
import math
import os
import pandas as pd

# -------------------------
# 1. Generate Temperature and Voltage
# -------------------------
np.random.seed(123)
n_pairs = 50
Temperature = np.random.uniform(250, 350, n_pairs)   # Example: 250-350 K
Voltage = np.random.uniform(11, 23, n_pairs)         # Example: 11-23 V
SIMULATION_BASE_SEED = 10000
# -------------------------
# 2. Define parameter function
# -------------------------

parameter_settings = {
    "baseline": {
        "a": (10, 140, -3),
        "c": (7, 125, -2),
    },
    "weaker_temperature": {
        "a": (10, 35, -3),
        "c": (7, 30, -2),
    },
    "positive_voltage": {
        "a": (1, 140, 0.3),
        "c": (1, 125, 0.2),
    },
    "weaker_stress": {
        "a": (2, 35, -0.1),
        "c": (2, 30, -0.1),
    },
}


def def_param(temp, vol):
    alpha = c0 + c1 / temp + c2 * np.log(vol)
    lam = a0 + a1 / temp + a2 * np.log(vol)
    return alpha, lam




# -------------------------
# 3. Weibull sampling under PHC
# -------------------------
def samp_weibull_PHC(n, R, alpha, lambd):
    R = np.array(R)
    m = len(R)
    r = np.random.rand(m)              # U(0,1)
    v = -np.log(1 - r)                 # Exp(1)
    X = np.zeros(m)
    denom = np.zeros(m)
    
    denom[0] = n
    X[0] = v[0] / denom[0]
    
    for i in range(1, m):
        denom[i] = denom[0] - i - np.sum(R[:i])
        X[i] = X[i-1] + (v[i] / denom[i])
    
    p = 1 - np.exp(-X)
    Y = weibull_min.ppf(p, c=alpha, scale=lambd)  # qweibull equivalent
    return Y


# -------------------------
# 4. Objective function for PHC
# -------------------------
def PHC_objfun(x_dat, R, t, n, beta):
    x_dat = np.sort(np.ravel(x_dat))
    R = np.ravel(R)
    m = len(R)
    
    j = np.sum(x_dat < t)
    
    if j == 0:
        func = 1
    elif j > 0 and j < m:
        lambd = j / (np.sum((1+R[:j]) * (x_dat[:j]**beta)) + (n - j - np.sum(R[:j])) * t**beta)
        term1 = np.sum(np.log(x_dat[:j]))
        term2 = (np.sum((1+R[:j]) * (x_dat[:j]**beta) * np.log(x_dat[:j])) 
                 + (n - j - np.sum(R[:j])) * np.log(t) * t**beta)
        func = j/beta + term1 - lambd*term2
    else:  # j == m
        lambd = m / np.sum((1+R) * (x_dat**beta))
        term1 = np.sum(np.log(x_dat))
        term2 = np.sum((1+R) * np.log(x_dat) * (x_dat**beta))
        func = m/beta + term1 - lambd*term2
    
    return func


# -------------------------
# 5. Derivative of estimating equation
# -------------------------
def beta_est_deriv(x_dat, R, t, n, beta):
    x_dat = np.sort(np.ravel(x_dat))
    R = np.ravel(R)
    m = len(R)
    
    j = np.sum(x_dat < t)
    
    if j < m:
        term0 = (np.sum((1+R[:j]) * (x_dat[:j]**beta)) 
                 + (n - j - np.sum(R[:j])) * t**beta)
        term1 = (np.sum((np.log(x_dat[:j])**2) * (x_dat[:j]**beta) * (1+R[:j])) 
                 + (n - j - np.sum(R[:j])) * t**beta * (np.log(t)**2))
        term2 = (np.sum((1+R[:j]) * (x_dat[:j]**beta) * np.log(x_dat[:j])) 
                 + (n - j - np.sum(R[:j])) * np.log(t) * t**beta)
        func = -j/(beta**2) - j*(term1*term0 - term2**2)/(term0**2)
    else:
        term0 = np.sum((1+R) * (x_dat**beta))
        term1 = np.sum((np.log(x_dat)**2) * (1+R) * (x_dat**beta))
        term2 = np.sum((1+R) * np.log(x_dat) * (x_dat**beta))
        func = -m/(beta**2) - m*(term1*term0 - term2**2)/(term0**2)
    
    return func


# -------------------------
# 6. Solve beta using Newton–Raphson
# -------------------------

def solve_beta_PHC(x_dat, R, t, n, max_iter=100, tol=1e-8):
    if np.sum(np.asarray(x_dat) < t) == 0:
        return np.nan

    root = 1.0

    for _ in range(max_iter):
        with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
            f_val = PHC_objfun(x_dat, R, t, n, root)
            f_deriv = beta_est_deriv(x_dat, R, t, n, root)

        if (
            not np.isfinite(f_val)
            or not np.isfinite(f_deriv)
            or f_deriv == 0
        ):
            return np.nan

        new_root = root - f_val / f_deriv

        if not np.isfinite(new_root):
            return np.nan

        # Reduce the step if Newton proposes a nonpositive shape.
        if new_root <= 0:
            new_root = root / 2

        if abs(new_root - root) <= tol * (1 + abs(root)):
            with np.errstate(
                over="ignore", divide="ignore", invalid="ignore"
            ):
                score = PHC_objfun(x_dat, R, t, n, new_root)

            if np.isfinite(score) and abs(score) <= 1e-6:
                return new_root

        root = new_root

    # Record nonconvergence instead of using an unchecked estimate.
    return np.nan


# -------------------------
# 7. Solve lambda
# -------------------------
def solve_lambda_PHC(x_dat, R, t, n, beta=None):
    if beta is None:
        beta = solve_beta_PHC(x_dat, R, t, n)
        
    x_dat = np.sort(np.ravel(x_dat))
    R = np.ravel(R)
    m = len(R)
    j = np.sum(x_dat < t)
    if j == 0 or not np.isfinite(beta) or beta <= 0:
        return np.nan
    
    if j < m:
        term = np.sum((1+R[:j]) * (x_dat[:j]**beta)) + (n - j - np.sum(R[:j])) * t**beta
        lambd = (term/j)**(1/beta)
    else:
        term = np.sum((1+R) * (x_dat**beta))
        lambd = (term/m)**(1/beta)
    
    return lambd



# -------------------------
# 7A. Approximate 95% and 99% confidence intervals
# -------------------------

def confidence_intervals_PHC(x_dat, R, t, n, beta, lambd):
    x_dat = np.sort(np.ravel(x_dat))
    R = np.asarray(R)
    m = len(R)
    d = int(np.sum(x_dat < t))

    if (
        d == 0
        or not np.isfinite(beta)
        or not np.isfinite(lambd)
        or beta <= 0
        or lambd <= 0
    ):
        raise ValueError("Invalid MLE for confidence intervals.")

    if d < m:
        times = np.concatenate([x_dat[:d], [t]])
        weights = np.concatenate([
            1 + R[:d],
            [n - d - np.sum(R[:d])],
        ])
    else:
        times = x_dat
        weights = 1 + R

    log_ratio = np.log(times) - np.log(lambd)
    h = beta * log_ratio

    with np.errstate(over="ignore", invalid="ignore"):
        q = weights * np.exp(h)

        # Negative Hessian in (log(beta), log(lambd)).
        information = np.array([
            [
                -beta * np.sum(np.log(x_dat[:d]) - np.log(lambd))
                + np.sum(q * (h**2 + h)),
                beta * (d - np.sum(q * (1 + h))),
            ],
            [
                beta * (d - np.sum(q * (1 + h))),
                beta**2 * np.sum(q),
            ],
        ])

    if not np.all(np.isfinite(information)):
        raise ValueError("Nonfinite information matrix.")

    np.linalg.cholesky(information)
    covariance = np.linalg.inv(information)

    se_log_beta, se_log_lambda = np.sqrt(np.diag(covariance))

    intervals = {}

    for level, z in (
        (95, 1.959963984540054),
        (99, 2.5758293035489004),
    ):
        with np.errstate(over="ignore", under="ignore"):
            endpoints = np.exp([
                np.log(beta) - z * se_log_beta,
                np.log(beta) + z * se_log_beta,
                np.log(lambd) - z * se_log_lambda,
                np.log(lambd) + z * se_log_lambda,
            ])

        if np.any(~np.isfinite(endpoints)) or np.any(endpoints <= 0):
            raise ValueError("Invalid confidence interval endpoints.")

        intervals[f"Alpha_CI{level}_lower"] = endpoints[0]
        intervals[f"Alpha_CI{level}_upper"] = endpoints[1]
        intervals[f"Lambda_CI{level}_lower"] = endpoints[2]
        intervals[f"Lambda_CI{level}_upper"] = endpoints[3]

    return intervals
# --------------------------
# 8. Load all datasets
# --------------------------

from Theoretical_Datasets import datasets
# If your filename is theoraticaldatasets.py, use:
# from theoraticaldatasets import datasets

n_sim = 1000

output_root = os.path.join("Results", "PHC")
print("Results location:", os.path.abspath(output_root), flush=True)
os.makedirs(output_root, exist_ok=True)


ci_columns = [
    f"{parameter}_CI{level}_{endpoint}"
    for parameter in ("Alpha", "Lambda")
    for level in (95, 99)
    for endpoint in ("lower", "upper")
]

all_summary_rows = []

# --------------------------
# 9. Run every dataset sequentially
# --------------------------
# --------------------------
# 9. Run all parameter settings and datasets
# --------------------------

scenarios = [
    (parameter_name, coefficients, dataset_name, setting)
    for parameter_name, coefficients in parameter_settings.items()
    for dataset_name, setting in datasets.items()
]

total_scenarios = len(scenarios)

print(
    f"Running {len(parameter_settings)} parameter settings "
    f"x {len(datasets)} datasets = {total_scenarios} scenarios.\n"
    f"Total simulated experiments: "
    f"{total_scenarios * n_pairs * n_sim:,}",
    flush=True,
)
# ============================================================
# BATCH SETTINGS
# ============================================================
START_PAIR = 25
END_PAIR = 26 
# Run batches as:
# 0,10   -> pairs 1-10
# 10,20  -> pairs 11-20
# 20,30  -> pairs 21-30
# 30,40  -> pairs 31-40
# 40,50  -> pairs 41-50


for scenario_number, (
    parameter_name,
    coefficients,
    dataset_name,
    setting,
) in enumerate(scenarios, start=1):

    # Select the coefficients for this scenario.
    a0, a1, a2 = coefficients["a"]
    c0, c1, c2 = coefficients["c"]
    parameter_index = list(parameter_settings.keys()).index(parameter_name)
    dataset_index = list(datasets.keys()).index(dataset_name)

    # Use the same temperature–voltage pairs for every scenario.
    params = [
        def_param(T, V)
        for T, V in zip(Temperature, Voltage)
    ]
    alpha = np.array([p[0] for p in params])
    lam = np.array([p[1] for p in params])

    if (
        np.any(~np.isfinite(alpha))
        or np.any(~np.isfinite(lam))
        or np.any(alpha <= 0)
        or np.any(lam <= 0)
    ):
        raise ValueError(
            f"Invalid true parameters for {parameter_name}."
        )

    n = setting["n"]
    R = np.asarray(setting["R"])
    t = setting["t"]
    m = len(R)

    assert m + np.sum(R) == n, f"Invalid scheme: {dataset_name}"
    assert np.all(R >= 0), f"Negative removals: {dataset_name}"

    print(
        f"\nScenario {scenario_number}/{total_scenarios}: "
        f"{parameter_name} | {dataset_name} | "
        f"n={n}, m={m}, m/n={m/n:.2%}",
        flush=True,
    )

    base_output_dir = os.path.join(
        output_root,
        parameter_name,
    )

    output_dir = os.path.join(
        base_output_dir,
        dataset_name,
    )
    lambda_hist_dir = os.path.join(
        output_dir, "Histograms", "Lambda"
    )
    alpha_hist_dir = os.path.join(
        output_dir, "Histograms", "Alpha"
    )
    replicate_dir = os.path.join(output_dir, "Replicates")
    data_dir = os.path.join(output_dir, "Simulated_data")

    for folder in (
        output_dir,
        lambda_hist_dir,
        alpha_hist_dir,
        replicate_dir,
        data_dir,
    ):
        os.makedirs(folder, exist_ok=True)

    pd.DataFrame([{
        "Dataset": dataset_name,
        "Parameter_setting": parameter_name,
        "a0": a0,
        "a1": a1,
        "a2": a2,
        "c0": c0,
        "c1": c1,
        "c2": c2,
        "n": n,
        "m": m,
        "t": t,
        "N_pairs": n_pairs,
        "N_simulations": n_sim,
        "Seed": 123,
    }]).to_csv(
        os.path.join(output_dir, "Settings.csv"),
        index=False,
    )

    # ------------------------------------------------------------
    # Load already completed summary results for this scenario
    # ------------------------------------------------------------
    
    summary_file = os.path.join(
        output_dir,
        "True_vals_&_MLE.csv"
    )
    
    if os.path.exists(summary_file):
        old_summary_df = pd.read_csv(summary_file)
    
        # Keep only valid pair numbers and avoid duplicates
        if "Pair" in old_summary_df.columns:
            old_summary_df = old_summary_df.drop_duplicates(
                subset=["Pair"],
                keep="last"
            )
    
        summary_rows = old_summary_df.to_dict("records")
    
    else:
        summary_rows = []

    # --------------------------
    # 10. Simulate each stress pair
    # --------------------------
    
    for j in range(START_PAIR, END_PAIR):
        # --------------------------------------------------------
        # Deterministic seed for this parameter/dataset/pair
        # --------------------------------------------------------
        pair_seed = (
            SIMULATION_BASE_SEED
            + parameter_index * 1_000_000
            + dataset_index * 10_000
            + j
        )
    
        np.random.seed(pair_seed)
        alpha_est = np.full(n_sim, np.nan)
        lambda_est = np.full(n_sim, np.nan)

        observed_counts = np.zeros(n_sim, dtype=int)
        stop_times = np.zeros(n_sim)

        observed_times = np.full((n_sim, m), np.nan)

        ci_results = {
            name: np.full(n_sim, np.nan)
            for name in ci_columns
        }

        fit_success = np.zeros(n_sim, dtype=bool)
        ci_success = np.zeros(n_sim, dtype=bool)

        for i in range(n_sim):
            x_dat = samp_weibull_PHC(
                n, R, alpha[j], lam[j]
            )

            d = int(np.sum(x_dat < t))
            observed_counts[i] = d
            observed_times[i, :d] = x_dat[:d]
            stop_times[i] = t if d < m else x_dat[-1]

            with np.errstate(
                over="ignore",
                divide="ignore",
                invalid="ignore",
            ):
                beta_hat = solve_beta_PHC(x_dat, R, t, n)

                if not np.isfinite(beta_hat) or beta_hat <= 0:
                    continue

                lambda_hat = solve_lambda_PHC(
                    x_dat, R, t, n, beta=beta_hat
                )

            if not np.isfinite(lambda_hat) or lambda_hat <= 0:
                continue

            alpha_est[i] = beta_hat
            lambda_est[i] = lambda_hat
            fit_success[i] = True

            try:
                intervals = confidence_intervals_PHC(
                    x_dat, R, t, n, beta_hat, lambda_hat
                )

                for name, value in intervals.items():
                    ci_results[name][i] = value

                ci_success[i] = True

            except (
                ValueError,
                FloatingPointError,
                np.linalg.LinAlgError,
            ):
                # Keep the valid MLE even if its CI is unavailable.
                pass

        error_alpha = alpha_est - alpha[j]
        error_lambda = lambda_est - lam[j]

        # --------------------------
        # Save individual estimates and CIs
        # --------------------------

        replicate_df = pd.DataFrame({
            "Dataset": dataset_name,
            "Parameter_setting": parameter_name,
            "Pair": j + 1,
            "Repetition": np.arange(1, n_sim + 1),
            "Temperature": Temperature[j],
            "Voltage": Voltage[j],
            "Alpha_true": alpha[j],
            "Lambda_true": lam[j],
            "Alpha_est": alpha_est,
            "Lambda_est": lambda_est,
            "Error_alpha": error_alpha,
            "Error_lambda": error_lambda,
            "Observed_failures": observed_counts,
            "Stop_time": stop_times,
            "Fit_success": fit_success,
            "CI_success": ci_success,
            **ci_results,
        })

        replicate_df.to_csv(
            os.path.join(replicate_dir, f"pair_{j+1}.csv"),
            index=False,
        )

        # Save the same observed data for future FIML benchmarking.
        np.savez_compressed(
            os.path.join(data_dir, f"pair_{j+1}.npz"),
            observed_times=observed_times,
            observed_counts=observed_counts,
            stop_times=stop_times,
            n=n,
            R=R,
            t=t,
            temperature=Temperature[j],
            voltage=Voltage[j],
            alpha_true=alpha[j],
            lambda_true=lam[j],
        )

        # --------------------------
        # Compute summary statistics
        # --------------------------

        summary = {
            "Dataset": dataset_name,
            "Parameter_setting": parameter_name,
            "Pair": j + 1,
            "n": n,
            "m": m,
            "Planned_observed_fraction": m / n,
            "t": t,
            "Temperature": Temperature[j],
            "Voltage": Voltage[j],
            "Alpha_true": alpha[j],
            "Lambda_true": lam[j],
            "N_simulations": n_sim,
            "N_valid_fits": int(fit_success.sum()),
            "N_valid_CIs": int(ci_success.sum()),
            "Fit_failure_rate": 1 - fit_success.mean(),
            "CI_failure_rate": 1 - ci_success.mean(),
            "Mean_observed_failures": observed_counts.mean(),
            "Mean_observed_fraction": observed_counts.mean() / n,
            "Zero_failure_rate": np.mean(observed_counts == 0),
        }

        for parameter, estimates, true_value, short_name in (
            ("Alpha", alpha_est, alpha[j], "alpha"),
            ("Lambda", lambda_est, lam[j], "lambda"),
        ):
            valid_estimates = estimates[fit_success]
            errors = valid_estimates - true_value

            summary[f"{parameter} MLE"] = (
                valid_estimates.mean() if errors.size else np.nan
            )
            summary[f"SD_{short_name}"] = (
                valid_estimates.std(ddof=0)
                if errors.size else np.nan
            )
            summary[f"Bias_{short_name}"] = (
                errors.mean() if errors.size else np.nan
            )
            summary[f"RMSE_{short_name}"] = (
                np.sqrt(np.mean(errors**2))
                if errors.size else np.nan
            )

            for level in (95, 99):
                lower = ci_results[
                    f"{parameter}_CI{level}_lower"
                ]
                upper = ci_results[
                    f"{parameter}_CI{level}_upper"
                ]

                covered = (
                    ci_success
                    & (lower <= true_value)
                    & (true_value <= upper)
                )

                # Coverage among valid confidence intervals.
                summary[
                    f"Coverage{level}_{short_name}_valid"
                ] = (
                    covered[ci_success].mean()
                    if ci_success.any()
                    else np.nan
                )

                summary[
                    f"Mean_width{level}_{short_name}"
                ] = (
                    (upper[ci_success] - lower[ci_success]).mean()
                    if ci_success.any()
                    else np.nan
                )

                # Includes all attempted repetitions in denominator.
                summary[
                    f"Covering_CI{level}_{short_name}_all"
                ] = covered.mean()

        # ------------------------------------------------------------
        # Add/update this pair in the scenario summary
        # ------------------------------------------------------------
        
        # Remove an old result for this pair if it already exists.
        # This makes rerunning a batch safe.
        summary_rows = [
            row for row in summary_rows
            if int(row["Pair"]) != (j + 1)
        ]
        
        summary_rows.append(summary)
        
        # Keep pairs ordered 1, 2, 3, ..., 50
        summary_rows = sorted(
            summary_rows,
            key=lambda row: int(row["Pair"])
        )
        
        pd.DataFrame(summary_rows).to_csv(
            os.path.join(output_dir, "True_vals_&_MLE.csv"),
            index=False,
        )

        # --------------------------
        # Save error histograms
        # --------------------------

        for parameter, errors, folder in (
            ("Lambda", error_lambda, lambda_hist_dir),
            ("Alpha", error_alpha, alpha_hist_dir),
        ):
            valid_errors = errors[np.isfinite(errors)]

            if valid_errors.size:
                plt.figure()
                plt.hist(
                    valid_errors,
                    bins=50,
                    color="green",
                    alpha=0.7,
                )
                plt.title(
                    f"{parameter} Error Histogram - Pair {j+1}"
                )
                plt.xlabel("Estimate minus true value")
                plt.ylabel("Frequency")

                plt.savefig(
                    os.path.join(
                        folder,
                        f"{parameter.lower()}_error_hist_pair{j+1}.png",
                    )
                )
                plt.close()

        print(
            f"  Pair {j+1}/{n_pairs}: "
            f"valid estimates={fit_success.sum()}/{n_sim}, "
            f"valid CIs={ci_success.sum()}/{n_sim}",
            flush=True,
        )

    # ============================================================
    # Dataset complete
    # Rebuild combined summaries from all completed scenario files
    # ============================================================
    
    all_completed_dfs = []
    
    for p_name in parameter_settings.keys():
    
        parameter_dfs = []
    
        for d_name in datasets.keys():
    
            scenario_summary_file = os.path.join(
                output_root,
                p_name,
                d_name,
                "True_vals_&_MLE.csv"
            )
    
            if os.path.exists(scenario_summary_file):
    
                temp_df = pd.read_csv(
                    scenario_summary_file
                )
    
                if not temp_df.empty:
    
                    # Prevent duplicate pairs if something was rerun
                    if "Pair" in temp_df.columns:
                        temp_df = temp_df.drop_duplicates(
                            subset=["Pair"],
                            keep="last"
                        )
    
                    parameter_dfs.append(temp_df)
                    all_completed_dfs.append(temp_df)
    
        # --------------------------------------------------------
        # All datasets summary for this parameter setting
        # --------------------------------------------------------
        if parameter_dfs:
    
            parameter_combined_df = pd.concat(
                parameter_dfs,
                ignore_index=True
            )
    
            parameter_combined_df = parameter_combined_df.sort_values(
                ["Dataset", "Pair"]
            )
    
            parameter_combined_df.to_csv(
                os.path.join(
                    output_root,
                    p_name,
                    "All_datasets_summary.csv"
                ),
                index=False
            )
    
    
    # ------------------------------------------------------------
    # Combined results across ALL completed parameter settings
    # ------------------------------------------------------------
    
    if all_completed_dfs:
    
        combined_df = pd.concat(
            all_completed_dfs,
            ignore_index=True
        )
    
        combined_df = combined_df.sort_values(
            ["Parameter_setting", "Dataset", "Pair"]
        )
    
        combined_df.to_csv(
            os.path.join(
                output_root,
                "All_parameters_summary.csv"
            ),
            index=False
        )

print("\nAll datasets completed.", flush=True)



Results location: C:\Users\NetComp\Documents\RJ\Results\PHC
Running 4 parameter settings x 50 datasets = 200 scenarios.
Total simulated experiments: 10,000,000

Scenario 1/200: baseline | m25 | n=58, m=25, m/n=43.10%
  Pair 26/50: valid estimates=1000/1000, valid CIs=1000/1000

Scenario 2/200: baseline | m25n | n=75, m=25, m/n=33.33%
  Pair 26/50: valid estimates=1000/1000, valid CIs=1000/1000

Scenario 3/200: baseline | m25nm | n=58, m=25, m/n=43.10%
  Pair 26/50: valid estimates=1000/1000, valid CIs=1000/1000

Scenario 4/200: baseline | m40 | n=80, m=40, m/n=50.00%
  Pair 26/50: valid estimates=1000/1000, valid CIs=1000/1000

Scenario 5/200: baseline | m40n | n=120, m=40, m/n=33.33%
  Pair 26/50: valid estimates=1000/1000, valid CIs=1000/1000

Scenario 6/200: baseline | m40nm | n=80, m=40, m/n=50.00%
  Pair 26/50: valid estimates=1000/1000, valid CIs=1000/1000

Scenario 7/200: baseline | m50 | n=100, m=50, m/n=50.00%
  Pair 26/50: valid estimates=1000/1000, valid CIs=1000/1000

Scena